# `crypto(base, start, end=None, quote="USD")`

Daily prices for the coins `crypto.prices` holds, from FMP, in USD or any currency `forex()` can price. Change the arguments and re-run any cell.

> **About the data shown.** This function's data comes from FMP's API, and FMP's license doesn't allow displaying it publicly. So the outputs saved here are synthetic: made-up data from [`_synthetic.py`](_synthetic.py), run through the function's own, unchanged code. The values are not real prices or yields.
>
> To see real data, start Jupyter with `ARORA_REAL_DATA=1`: every cell then reads prod, read-only, as `readonly_user`. Don't commit those outputs; `tests/test_notebooks_offline.py` fails if you save them.

In [1]:
# Synthetic data in place of FMP's (see the note above), unless Jupyter was started with ARORA_REAL_DATA=1.
import _synthetic

_synthetic.use()

Data: synthetic


In [2]:
from datetime import date, timedelta

import polars as pl
from IPython.display import Markdown

from arora_data.crypto import NAMES, CryptoBar
from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.sync import crypto, list_series


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The coins
Read from the table on every call, so a coin the main repo adds or drops shows up here with no code change. The fixed top 15 have names; any other stored coin works by code, with `name` blank.

In [3]:
pl.DataFrame([{"coin": s.key.removesuffix("USD"), "name": NAMES.get(s.key.removesuffix("USD")), "latest_date": s.latest_date}
              for s in list_series("crypto", "prices")])

coin,name,latest_date
str,str,date
"""ADA""","""cardano""",2026-11-23
"""BNB""","""bnb""",2026-11-23
"""BTC""","""bitcoin""",2026-11-23
"""DOGE""","""dogecoin""",2026-11-23
"""ETH""","""ethereum""",2026-11-23
…,…,…
"""USDC""","""usd coin""",2026-11-23
"""USDT""","""tether""",2026-11-23
"""XMR""","""monero""",2026-11-23


## By code or name
A name works anywhere a code does, in any case and spacing. Every bar carries the code and the name.

In [4]:
week = today_et() - timedelta(days=7)
assert crypto("BTC", week) == crypto("bitcoin", week) == crypto("  Bitcoin ", week)
show(crypto("ETH", week)[-2:] + crypto("unus sed leo", week)[-2:])

4 rows

coin|name|quote|date|route|quote_date|open|high|low|close|volume_usd|vwap
---|---|---|---|---|---|---|---|---|---|---|---
ETH|ethereum|USD|2026-11-22|stored|None|88.198783|88.382575|87.726803|88.219982|7130|88.109787
ETH|ethereum|USD|2026-11-23|stored|None|87.883421|88.998512|87.147406|88.751534|8999|88.299151
LEO|unus sed leo|USD|2026-11-22|stored|None|131.535272|132.515597|129.541742|129.915757|3362|130.657699
LEO|unus sed leo|USD|2026-11-23|stored|None|129.571653|130.044584|128.396989|128.437695|3377|128.959756

## In USD, the default
FMP daily bars as stored (`route` = stored): open, high, low, close, `volume_usd` and VWAP. Crypto trades every day, weekends included. `quote_date` is blank because no rate is used.

In [5]:
show(crypto("BTC", today_et() - timedelta(days=7)))

8 rows

coin|name|quote|date|route|quote_date|open|high|low|close|volume_usd|vwap
---|---|---|---|---|---|---|---|---|---|---|---
BTC|bitcoin|USD|2026-11-16|stored|None|96.816525|99.526087|96.248787|98.338852|3382|98.037909
BTC|bitcoin|USD|2026-11-17|stored|None|98.290442|98.956848|97.870964|98.815025|7570|98.547612
BTC|bitcoin|USD|2026-11-18|stored|None|98.528655|100.650671|97.930247|99.556794|1150|99.379237
BTC|bitcoin|USD|2026-11-19|stored|None|99.195030|100.188967|98.762236|100.143663|9208|99.698289
BTC|bitcoin|USD|2026-11-20|stored|None|100.403479|101.589514|100.359461|101.394806|1782|101.114594
BTC|bitcoin|USD|2026-11-21|stored|None|101.283649|104.354653|100.882109|103.292401|5607|102.843055
BTC|bitcoin|USD|2026-11-22|stored|None|103.179021|105.274704|102.159306|104.832165|8553|104.088725
BTC|bitcoin|USD|2026-11-23|stored|None|104.718446|108.331132|104.611735|107.557841|7983|106.833570

## In another currency
The USD price times `forex("USD", "EUR")` that day (`route` = crossed). Open uses the day's opening rate, close its closing rate. High, low and VWAP are blank: the coin and the rate peak at different moments. `volume_usd` stays in US dollars.

In [6]:
show(crypto("SOL", today_et() - timedelta(days=7), quote="EUR"))

8 rows

coin|name|quote|date|route|quote_date|open|high|low|close|volume_usd|vwap
---|---|---|---|---|---|---|---|---|---|---|---
SOL|solana|EUR|2026-11-16|crossed|2026-11-16|121.124449870|None|None|119.866005422|4900|None
SOL|solana|EUR|2026-11-17|crossed|2026-11-17|119.776899139|None|None|120.255048697|9927|None
SOL|solana|EUR|2026-11-18|crossed|2026-11-18|119.607595849|None|None|116.788343916|2221|None
SOL|solana|EUR|2026-11-19|crossed|2026-11-19|116.648278677|None|None|116.676922125|3551|None
SOL|solana|EUR|2026-11-20|crossed|2026-11-20|116.628370684|None|None|114.078788575|1763|None
SOL|solana|EUR|2026-11-21|crossed|2026-11-20|114.478776208|None|None|112.192381112|1116|None
SOL|solana|EUR|2026-11-22|crossed|2026-11-20|112.208368851|None|None|113.674763152|3855|None
SOL|solana|EUR|2026-11-23|crossed|2026-11-23|113.817884883|None|None|111.288676256|8030|None

In [7]:
start = today_et() - timedelta(days=7)
rows = [r for q in ["USD", "EUR", "JPY", "GBP", "INR", "CHF"] for r in crypto("BTC", start, quote=q)]
to_df(rows, CryptoBar).pivot(on="quote", index="date", values="close")

date,USD,EUR,JPY,GBP,INR,CHF
date,"decimal[38,10]","decimal[38,10]","decimal[38,10]","decimal[38,10]","decimal[38,10]","decimal[38,10]"
2026-11-16,98.3388520000,88.8737930411,86.4255917745,88.1209620835,90.0783884320,105.0280573910
2026-11-17,98.8150250000,89.0949447209,87.1253063575,88.3765206410,90.0905476277,105.1564792290
2026-11-18,99.5567940000,89.3376400320,87.2886093890,89.1918932745,91.3235466930,106.2010153180
2026-11-19,100.1436630000,89.7770206487,87.7732167406,89.7424260755,92.1576064504,106.3021978440
2026-11-20,101.3948060000,89.5670316371,88.1748497989,90.1439407544,93.0550832065,106.9881490780
2026-11-21,103.2924010000,91.2432708658,89.8250344652,91.8309767872,94.7966010178,108.9904230090
2026-11-22,104.8321650000,92.6034203285,91.1640424951,93.1998870920,96.2097194288,110.6151265500
2026-11-23,107.5578410000,94.3151457065,93.2239302035,96.0721296803,98.0953323802,114.3714151120


## Days with no rate carry the last one
A day the currency pair has no bar, mostly a weekend, carries the pair's last close, for open and close alike, and `quote_date` names the day the rate came from. A day the pair does have a bar uses it. Here: carried days for ETH in EUR over the last year, by weekday.

In [8]:
eth_eur = to_df(crypto("ETH", today_et() - timedelta(days=365), quote="EUR"), CryptoBar)
carried = eth_eur.filter(pl.col("quote_date") != pl.col("date"))
print(f"{carried.height} of {eth_eur.height} days carried a rate")
carried.group_by(pl.col("date").dt.strftime("%a").alias("weekday")).len().sort("len", descending=True)

107 of 366 days carried a rate


weekday,len
str,u32
"""Sun""",53
"""Sat""",52
"""Thu""",2


## The first days look back before start
Starting on a carried day still gives that day a price: the rate is looked up as far as 14 days before `start`.

In [9]:
first = carried["date"][-1]
show(crypto("ETH", first, first + timedelta(days=2), quote="EUR"))

2 rows

coin|name|quote|date|route|quote_date|open|high|low|close|volume_usd|vwap
---|---|---|---|---|---|---|---|---|---|---|---
ETH|ethereum|EUR|2026-11-22|crossed|2026-11-20|77.9103338618|None|None|77.9290599838|7130|None
ETH|ethereum|EUR|2026-11-23|crossed|2026-11-23|77.8342420136|None|None|77.8243016321|8999|None

## `crypto("ALL", start, end, quote)`
Every stored coin in one currency, ordered by code then date. Each symbol is read once, and the currency rate once.

In [10]:
to_df(crypto("ALL", today_et() - timedelta(days=7)), CryptoBar).pivot(on="coin", index="date", values="close")

date,ADA,BNB,BTC,DOGE,ETH,HYPE,LEO,LINK,SOL,TRX,USDC,USDT,XMR,XRP,ZEC
date,"decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]"
2026-11-16,93.246461,105.946435,98.338852,135.695986,90.384300,103.138618,132.122513,97.300256,132.631735,75.121722,106.738874,86.235642,95.106901,111.881861,116.738572
2026-11-17,91.975053,105.316167,98.815025,135.673178,89.131601,102.984899,130.219406,99.380134,133.374634,77.104186,107.041370,85.965361,94.993009,113.630769,115.272669
2026-11-18,91.331218,105.311519,99.556794,133.952697,89.072923,101.304978,131.111085,98.304743,130.147529,78.747474,109.986996,85.301931,96.158200,116.797654,114.891724
2026-11-19,91.161317,105.050395,100.143663,133.739810,87.609520,99.773395,132.680822,98.859266,130.149723,78.931600,109.768427,86.835643,98.367296,114.879065,115.297525
2026-11-20,92.952319,105.850875,101.394806,134.639511,85.845271,97.759265,131.517081,100.447579,129.143463,78.862040,107.159938,86.408353,99.603611,114.606166,113.721656
2026-11-21,92.065565,105.997350,103.292401,133.815449,87.974757,97.565361,131.630099,100.096990,127.007946,78.427082,107.477406,84.302092,100.764932,113.846808,113.946167
2026-11-22,91.977282,105.287352,104.832165,131.796018,88.219982,99.932921,129.915757,101.234101,128.686084,78.730795,105.959606,85.555237,98.638004,113.298591,114.680606
2026-11-23,90.800417,104.620958,107.557841,131.701651,88.751534,97.662883,128.437695,102.173544,126.914608,77.016953,105.149859,85.682684,99.042642,111.786051,112.725566


## Full history in one call
Bitcoin in USD from the start of FMP's history (2010). There is no row limit.

In [11]:
btc = to_df(crypto("BTC", date(2009, 1, 1)), CryptoBar)
print(f"{btc.height:,} days, {btc['date'].min()} to {btc['date'].max()}")
(
    btc.group_by(pl.col("date").dt.year().alias("year"), maintain_order=True)
    .agg(pl.col("date").last().alias("last_day"), pl.col("close").last())
    .tail(10)
)

5,974 days, 2010-07-17 to 2026-11-23


year,last_day,close
i32,date,"decimal[38,6]"
2017,2017-12-31,123.048501
2018,2018-12-31,98.930072
2019,2019-12-31,112.159182
2020,2020-12-31,111.972206
2021,2021-12-31,110.570741
2022,2022-12-31,99.282274
2023,2023-12-31,99.197271
2024,2024-12-31,114.126447
2025,2025-12-31,88.238644


## Bad input
An FMP symbol instead of a code, a name that isn't on the list, `quote="ALL"`, a currency with no USD pair (a coin is not a quote currency), or a start after the end each raise a clear error.

In [12]:
week = today_et() - timedelta(days=7)
for base, quote in [("BTCUSD", "USD"), ("litecoin", "USD"), ("BTC", "ALL"), ("BTC", "XAU"), ("BTC", "ETH")]:
    try:
        crypto(base, week, quote=quote)
    except ValueError as err:
        print(f"{base}/{quote}: {err}")
try:
    crypto("BTC", today_et(), week)
except ValueError as err:
    print(f"start after end: {err}")

BTCUSD/USD: BTCUSD is not a stored coin: use a code or name, such as BTC or bitcoin
litecoin/USD: LITECOIN is not a stored coin: use a code or name, such as BTC or bitcoin
BTC/ALL: quote must be one currency: "ALL" is for base only
BTC/XAU: XAU has no stored USD pair, so USD/XAU can't be computed
BTC/ETH: ETH has no stored USD pair, so USD/ETH can't be computed
start after end: start must be on or before end
